In [ ]:
# 1. Inspect the remote runtime (never print credentials)
import sys, os, json, time, importlib.util
from pathlib import Path
import torch
print('Python:', sys.version.split()[0])
print('Working directory:', Path.cwd())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print('man.env exists:', Path('man.env').is_file())
print('Packages:', {p: importlib.util.find_spec(p) is not None for p in ['diffusers','transformers','accelerate','dotenv','typesafe_sdk','sklearn']})

In [ ]:
# 2. Install the experiment dependencies into this notebook kernel
%pip -q install 'diffusers==0.32.2' 'transformers==4.48.3' 'accelerate>=0.34,<2' safetensors python-dotenv scikit-learn scipy matplotlib pandas requests
print('Dependency installation finished.')

# Closed-loop diffusion: a falsifiable pilot

Goal: make a cathedral-like organism more biological while preserving structure and background.

Stable Diffusion 1.5 + deterministic DDIM, DINOv2 with registers for spatial regions, CLIP for semantic proxies, and TypeSafe/Jev for decisions over measured three-step counterfactual branches. Compare matched seeds: baseline, fixed regional guidance, deterministic control, and Jev control.

This implements the register-assisted EXTERNAL targeting variant in the notes. SD1.5 has no internal registers; this does not test individual register causality or prove novelty. DINO clusters are candidate regions, not guaranteed object masks. CLIP cosine scores are proxies, not probabilities. Three seeds are a feasibility pilot.

Sources: https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5 ; https://huggingface.co/docs/transformers/model_doc/dinov2_with_registers ; https://docs.typesafe.ai/api

Only synthetic metrics go to TypeSafe. Credentials are never printed or saved in results.

In [ ]:
# 3. Imports, safe credential loading, and reproducible settings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch.nn.functional as F
from PIL import Image
from scipy.ndimage import gaussian_filter, sobel
from sklearn.cluster import KMeans
from dotenv import dotenv_values
from diffusers import StableDiffusionPipeline, DDIMScheduler
from transformers import AutoImageProcessor, AutoModel, CLIPModel, CLIPProcessor
import requests, importlib.metadata as metadata
from IPython.display import display, Markdown
_env = dotenv_values('man.env')
_keys = [v for k,v in _env.items() if v and any(s in k.upper() for s in ('TYPESAFE','JEV'))]
if not _keys:
    _keys = [v for k,v in _env.items() if v and 'KEY' in k.upper()]
assert len(_keys) == 1, 'Expected one unambiguous TypeSafe API key in man.env.'
_api_key = _keys[0]
del _env, _keys
DEVICE, DTYPE = 'cuda', torch.float16
SEEDS, STEPS, SIZE, CFG = [17, 42, 123], 36, 512, 7.5
CHECKPOINTS, HORIZON = [9, 15, 21, 27], 3
OUT = Path('results') / time.strftime('closed_loop_%Y%m%d_%H%M%S')
OUT.mkdir(parents=True, exist_ok=True)
VERSIONS = {p: metadata.version(p) for p in ['torch','diffusers','transformers','numpy','scikit-learn']}
print('Credentials loaded:', bool(_api_key), '(value hidden)')
print('Versions:', VERSIONS)
print('Outputs:', OUT)

In [ ]:
# 4. Load the actual pretrained diffusion model on the A40
MODEL_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
t0 = time.time()
pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE, use_safetensors=True).to(DEVICE)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.scheduler.set_timesteps(STEPS, device=DEVICE)
pipe.set_progress_bar_config(disable=True)
pipe.unet.eval(); pipe.vae.eval(); pipe.text_encoder.eval()
print('Loaded SD1.5 in', round(time.time()-t0, 1), 'seconds')
print('Scheduler:', type(pipe.scheduler).__name__, '| prediction:', pipe.scheduler.config.prediction_type)
print('GPU allocated GiB:', round(torch.cuda.memory_allocated()/2**30, 2))

In [ ]:
# 5. Fixed conditions and a transparent DDIM stepping function
BASE_PROMPT = 'A single monumental gothic cathedral, symmetrical spires and ribbed arches, standing alone on a barren plain, surreal architectural concept art, intricate detail, dramatic overcast light, wide view'
BIO_PROMPT = 'A single living cathedral organism, gothic spires grown from ivory bone, translucent fleshy membranes and branching organic tendons, standing alone on a barren plain, surreal architectural concept art, intricate detail, dramatic overcast light, wide view'
NEGATIVE = 'text, watermark, blurry, low quality, cropped, collage'
@torch.inference_mode()
def encode_text(texts):
    tokens = pipe.tokenizer(texts, padding='max_length', max_length=77, truncation=True, return_tensors='pt').to(DEVICE)
    return pipe.text_encoder(tokens.input_ids)[0]
emb = encode_text([NEGATIVE, BASE_PROMPT, BIO_PROMPT])
@torch.inference_mode()
def initial_latent(seed):
    return torch.randn((1,4,SIZE//8,SIZE//8), generator=torch.Generator(device=DEVICE).manual_seed(seed), device=DEVICE, dtype=DTYPE) * pipe.scheduler.init_noise_sigma
@torch.inference_mode()
def advance(z, i, masks=None, strengths=None):
    n = len(z); t = pipe.scheduler.timesteps[i]
    conditions = 3 if masks is not None else 2
    noise = pipe.unet(z.repeat(conditions,1,1,1), t, encoder_hidden_states=torch.cat([emb[k:k+1].repeat(n,1,1) for k in range(conditions)])).sample
    parts = noise.chunk(conditions)
    eps = parts[0] + CFG * (parts[1]-parts[0])
    if masks is not None:
        eps = eps + strengths[:,None,None,None] * masks * CFG * (parts[2]-parts[1])
    step = pipe.scheduler.step(eps, t, z, eta=0.0)
    return step.prev_sample, step.pred_original_sample
@torch.inference_mode()
def decode(z):
    images = (pipe.vae.decode(z / pipe.vae.config.scaling_factor).sample / 2 + 0.5).clamp(0,1)
    return [Image.fromarray((x*255).round().astype('uint8')) for x in images.float().cpu().permute(0,2,3,1).numpy()]
print('Conditions encoded:', tuple(emb.shape))
print('Intervention: eps_base + strength * spatial_mask * CFG * (eps_bio - eps_base_cond)')

In [ ]:
# 6. First real image: baseline, seed 17, plus cached checkpoints
baselines, baseline_paths = {}, {}
def generate_baseline(seed):
    z = initial_latent(seed); checkpoints = {}; start = time.time()
    for i in range(STEPS):
        if i in CHECKPOINTS: checkpoints[i] = z.clone()
        z, x0 = advance(z, i)
    image = decode(z)[0]
    image.save(OUT / f'{seed}_baseline.png')
    baselines[seed] = image; baseline_paths[seed] = checkpoints
    print('Baseline seed', seed, '| seconds', round(time.time()-start,1))
    return image
display(generate_baseline(SEEDS[0]))

In [ ]:
# 7. Load visual observers (pretrained weights, no remote model code)
DINO_ID = 'facebook/dinov2-with-registers-small'
CLIP_ID = 'openai/clip-vit-base-patch32'
dino_processor = AutoImageProcessor.from_pretrained(DINO_ID)
dino = AutoModel.from_pretrained(DINO_ID, use_safetensors=True).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)
clip = CLIPModel.from_pretrained(CLIP_ID, use_safetensors=True).to(DEVICE).eval()
CONCEPTS = ['a living biological organism made of bone and translucent organic membranes', 'a gothic stone cathedral with spires and arches', 'a barren landscape under an overcast sky']
with torch.inference_mode():
    concept_features = F.normalize(clip.get_text_features(**clip_processor(text=CONCEPTS, padding=True, return_tensors='pt').to(DEVICE)), dim=-1)
print('DINO registers:', dino.config.num_register_tokens, '| patch size:', dino.config.patch_size)
print('Observers ready; CLIP scores will be cosine similarities, not probabilities.')

In [ ]:
# 8. Compile perceptual context from actual image measurements
@torch.inference_mode()
def observe(images):
    pixels = dino_processor(images=images, return_tensors='pt').to(DEVICE)
    hidden = dino(**pixels).last_hidden_state
    patches = F.normalize(hidden[:,1+dino.config.num_register_tokens:], dim=-1)
    cf = F.normalize(clip.get_image_features(**clip_processor(images=images, return_tensors='pt').to(DEVICE)), dim=-1)
    scores = (cf @ concept_features.T).cpu().numpy()
    return patches.cpu().numpy(), scores
def find_regions(image, patches):
    side = int(np.sqrt(len(patches)))
    labels = KMeans(n_clusters=4, random_state=0, n_init=10).fit_predict(patches).reshape(side,side)
    yy,xx = np.mgrid[:side,:side] / (side-1)
    center = np.exp(-((xx-.5)**2+(yy-.48)**2)/.13)
    ranks = sorted(range(4), key=lambda k: float(center[labels==k].mean()), reverse=True)
    regions, masks, crops = {}, [], []
    for name,k in zip(['A','B'],ranks[:2]):
        hard = labels==k
        soft = gaussian_filter(hard.astype('float32'), .65)
        ys,xs = np.where(hard)
        box = (int(xs.min()/side*SIZE), int(ys.min()/side*SIZE), int((xs.max()+1)/side*SIZE), int((ys.max()+1)/side*SIZE))
        regions[name] = {'area':round(float(hard.mean()),4), 'center':[round(float(xx[hard].mean()),3),round(float(yy[hard].mean()),3)], 'bbox':list(box), 'label':'DINO patch cluster; identity not guaranteed across checkpoints'}
        masks.append(soft); crops.append(image.crop(box))
    _, crop_scores = observe(crops)
    for name,s in zip(regions,crop_scores):
        regions[name]['organic_cosine'] = round(float(s[0]),5)
        regions[name]['architecture_cosine'] = round(float(s[1]),5)
    mt = torch.tensor(np.stack(masks), device=DEVICE)[:,None]
    mt = F.interpolate(mt, size=(SIZE//8,SIZE//8), mode='bilinear', align_corners=False).to(DTYPE)
    return regions, mt, labels
def metrics(image, patch, scores, reference_image, reference_patch, foreground):
    arr = np.asarray(image,dtype='float32')/255
    ref = np.asarray(reference_image,dtype='float32')/255
    bg = 1-F.interpolate(foreground.float(),size=(SIZE,SIZE),mode='bilinear',align_corners=False)[0,0].cpu().numpy()
    gray = arr.mean(-1)
    return {'organic_cosine':float(scores[0]),'architecture_cosine':float(scores[1]),
            'dino_patch_similarity':float((patch*reference_patch).sum(-1).mean()),
            'background_mae':float((np.abs(arr-ref)*bg[...,None]).sum()/(3*bg.sum()+1e-8)),
            'edge_energy':float(np.hypot(sobel(gray,0),sobel(gray,1)).mean())}
print('Observer and context compiler ready.')

In [ ]:
# 9. Inspect what the controller can actually see at step 9
z9 = baseline_paths[17][CHECKPOINTS[0]]
_, x09 = advance(z9, CHECKPOINTS[0])
preview9 = decode(x09)[0]
p9, s9 = observe([preview9])
regions9, masks9, labels9 = find_regions(preview9, p9[0])
fig, axes = plt.subplots(1,4,figsize=(15,4))
axes[0].imshow(preview9); axes[0].set_title('Predicted clean image: step 9')
axes[1].imshow(labels9, cmap='tab10'); axes[1].set_title('DINO patch clusters')
for ax,m,name in zip(axes[2:],masks9,['Region A','Region B']):
    ax.imshow(preview9); ax.imshow(F.interpolate(m[None].float(),size=(SIZE,SIZE),mode='bilinear',align_corners=False)[0,0].cpu(),alpha=.5,cmap='magma',vmin=0,vmax=1); ax.set_title(name)
for ax in axes: ax.axis('off')
plt.tight_layout(); fig.savefig(OUT/'observer_regions.png',dpi=150,bbox_inches='tight'); plt.show()
display(pd.DataFrame(regions9).T)
print('Global CLIP cosines [organic, architecture, landscape]:', np.round(s9[0],4))

In [ ]:
# 10. Probe five counterfactual futures from the exact same latent
ACTION_NAMES = ['none','A_low','A_high','B_low','B_high']
ACTION_STRENGTHS = torch.tensor([0.,.45,.9,.45,.9], device=DEVICE, dtype=DTYPE)
@torch.inference_mode()
def probe(z, i, history):
    _, current_x0 = advance(z,i)
    current_image = decode(current_x0)[0]
    cp, cs = observe([current_image])
    regions, masks, labels = find_regions(current_image, cp[0])
    action_masks = torch.stack([torch.zeros_like(masks[0]),masks[0],masks[0],masks[1],masks[1]])
    branches = z.repeat(5,1,1,1)
    for j in range(i,min(i+HORIZON,STEPS)):
        branches, x0 = advance(branches,j,action_masks,ACTION_STRENGTHS)
    previews = decode(x0)
    patches, scores = observe(previews)
    foreground = masks.sum(0,keepdim=True).clamp(0,1)
    outcomes = {}
    for k,name in enumerate(ACTION_NAMES):
        m = metrics(previews[k],patches[k],scores[k],previews[0],patches[0],foreground)
        m['organic_gain'] = float(scores[k,0]-scores[0,0])
        m['architecture_change'] = float(scores[k,1]-scores[0,1])
        outcomes[name] = {key:round(value,6) for key,value in m.items()}
    current = {'organic_cosine':round(float(cs[0,0]),6),'architecture_cosine':round(float(cs[0,1]),6)}
    state = {'goal':'Increase biological material while moderately preserving gothic architecture and keeping background stable',
        'progress':round(i/STEPS,3),'phase':'early' if i<12 else ('middle' if i<24 else 'late'),
        'scene':current,'regions':regions,'candidate_outcomes':outcomes,
        'metric_guide':'organic_gain and architecture_change are CLIP cosine deltas versus the simultaneous no-action preview. DINO similarity 1 means unchanged features, not identical geometry. background_mae 0 means unchanged pixels outside the current foreground estimate. No metric is a calibrated semantic probability.',
        'recent_changes':{k:round(current[k]-history[-1]['scene'][k],6) for k in current} if history else {},
        'history':history[-2:]}
    return branches, previews, state, masks
def deterministic_scores(state):
    return {name: 100*m['organic_gain'] + 30*min(m['architecture_change'],0) - 5*(1-m['dino_patch_similarity']) - 5*m['background_mae'] for name,m in state['candidate_outcomes'].items()}
branches9, previews9, state9, _ = probe(z9,9,[])
display(pd.DataFrame(state9['candidate_outcomes']).T)
fig,axes=plt.subplots(1,5,figsize=(17,4))
for ax,name,img in zip(axes,ACTION_NAMES,previews9): ax.imshow(img); ax.set_title(name); ax.axis('off')
plt.tight_layout(); fig.savefig(OUT/'counterfactual_step9.png',dpi=150,bbox_inches='tight'); plt.show()
print('Deterministic utilities:', deterministic_scores(state9))

In [ ]:
# 11. TypeSafe: three narrow judgments over one measured state
API_LOG = []
QUESTIONS = {
 'material': {'type':'choice','instructions':'Which candidate has the clearest positive organic_gain in candidate_outcomes? Prefer none if all gains are below 0.001, since these tiny CLIP deltas are weak evidence. Judge only biological material evidence.','criteria':{n:'The measured '+n+' candidate in candidate_outcomes.' for n in ACTION_NAMES}},
 'structure': {'type':'choice','instructions':'Which candidate best preserves structure, using high dino_patch_similarity and nonnegative architecture_change in candidate_outcomes? Judge only structural preservation.','criteria':{n:'The measured '+n+' candidate in candidate_outcomes.' for n in ACTION_NAMES}},
 'background': {'type':'choice','instructions':'Which candidate best preserves the background, using lowest background_mae in candidate_outcomes? Judge only background preservation.','criteria':{n:'The measured '+n+' candidate in candidate_outcomes.' for n in ACTION_NAMES}}
}
def jev_route(state):
    start = time.time()
    for attempt in range(3):
        response = requests.post('https://api.typesafe.ai/v1/systemone',headers={'Authorization':'Bearer '+_api_key},json={'model':'jev-latest','state':state,'questions':QUESTIONS},timeout=60)
        if response.status_code not in (429,529): break
        time.sleep(2**attempt)
    if response.status_code != 200:
        raise RuntimeError('TypeSafe request failed with HTTP '+str(response.status_code)+'; response body withheld to protect credentials.')
    data = response.json(); answers = data['answers']
    for q in QUESTIONS:
        probs = answers[q]['probabilities']
        assert set(probs)==set(ACTION_NAMES) and abs(sum(probs.values())-1)<.02
    # Predeclared preference: material 60%, structure 25%, background 15%.
    routing = {n: .60*answers['material']['probabilities'][n]+.25*answers['structure']['probabilities'][n]+.15*answers['background']['probabilities'][n] for n in ACTION_NAMES}
    selected = max(routing,key=routing.get)
    audit = {'model':data.get('model'),'answers':answers,'routing_weights':routing,'selected':selected,'usage':data.get('usage'),'seconds':round(time.time()-start,3)}
    API_LOG.append(audit)
    return selected,audit
first_jev_action, first_jev_audit = jev_route(state9)
print('Actual TypeSafe model:', first_jev_audit['model'])
print('Selected:', first_jev_action, '| latency:', first_jev_audit['seconds'])
display(pd.DataFrame({q:first_jev_audit['answers'][q]['probabilities'] for q in QUESTIONS}).assign(aggregate=pd.Series(first_jev_audit['routing_weights'])))

In [ ]:
# 12. Closed-loop sampler: commit a measured branch, observe again, replan
RUNS = {}
def run_control(seed, mode):
    assert mode in ['fixed','deterministic','jev']
    start = time.time(); history=[]; decisions=[]; frames=[]
    z = baseline_paths[seed][CHECKPOINTS[0]].clone()
    i = CHECKPOINTS[0]; fixed_mask = None
    while i < STEPS:
        if i in CHECKPOINTS:
            if mode == 'fixed':
                if fixed_mask is None:
                    _,x0=advance(z,i); img=decode(x0)[0]; p,_=observe([img])
                    _,fm,_=find_regions(img,p[0]); fixed_mask=fm[0:1]
                for j in range(i,i+HORIZON):
                    z,x0=advance(z,j,fixed_mask,torch.tensor([.9],device=DEVICE,dtype=DTYPE))
                frames.append(decode(x0)[0]); decisions.append({'step':i,'selected':'A_high','policy':'fixed first-checkpoint region, fixed strength'})
            else:
                if seed==17 and i==9:
                    branches, previews, state = branches9.clone(),previews9,json.loads(json.dumps(state9))
                else:
                    branches,previews,state,_=probe(z,i,history)
                if mode=='deterministic':
                    utilities=deterministic_scores(state); action=max(utilities,key=utilities.get); audit={'utilities':utilities}
                else:
                    if seed==17 and i==9: action,audit=first_jev_action,first_jev_audit
                    else: action,audit=jev_route(state)
                k=ACTION_NAMES.index(action); z=branches[k:k+1].clone()
                frames.append(previews[k])
                decisions.append({'step':i,'selected':action,'state':state,'router':audit})
                history.append({'step':i,'scene':state['scene'],'action':action,'measured_effect_vs_no_action':state['candidate_outcomes'][action]})
                print(f'{mode} seed={seed} step={i}: {action}',flush=True)
            i += HORIZON
        else:
            z,_=advance(z,i); i+=1
    image=decode(z)[0]; image.save(OUT/f'{seed}_{mode}.png')
    elapsed=round(time.time()-start,2)
    result={'image':image,'decisions':decisions,'frames':frames,'seconds':elapsed}
    RUNS[(seed,mode)] = result
    (OUT/f'{seed}_{mode}_decisions.json').write_text(json.dumps({'seed':seed,'mode':mode,'seconds':elapsed,'decisions':decisions},indent=2))
    print(f'Completed {mode}, seed {seed}, {elapsed}s',flush=True)
    return image
print('Closed-loop sampler ready. Each selected future is actually committed; no latent averaging.')

In [ ]:
# 13. Matched-seed comparison: run each policy on seed 17
for mode in ['fixed','deterministic','jev']:
    run_control(17,mode)
fig,axes=plt.subplots(1,4,figsize=(18,5))
for ax,mode in zip(axes,['baseline','fixed','deterministic','jev']):
    ax.imshow(baselines[17] if mode=='baseline' else RUNS[(17,mode)]['image']); ax.set_title(mode); ax.axis('off')
plt.tight_layout(); fig.savefig(OUT/'seed17_comparison.png',dpi=160,bbox_inches='tight'); plt.show()

In [ ]:
# 14. Replicate the predeclared policies on two further seeds
for seed in SEEDS[1:]:
    generate_baseline(seed)
    for mode in ['fixed','deterministic','jev']:
        run_control(seed,mode)
print('Completed all',len(SEEDS)*4,'images. No seed or policy was discarded.')

In [ ]:
# 15. Final measurements: identical reference and foreground for every policy
MODES = ['baseline','fixed','deterministic','jev']
rows=[]
for seed in SEEDS:
    images=[baselines[seed]]+[RUNS[(seed,m)]['image'] for m in MODES[1:]]
    patches,scores=observe(images)
    _,reference_masks,_=find_regions(images[0],patches[0])
    fg=reference_masks.sum(0,keepdim=True).clamp(0,1)
    for k,mode in enumerate(MODES):
        row=metrics(images[k],patches[k],scores[k],images[0],patches[0],fg)
        row.update(seed=seed,mode=mode,organic_gain=float(scores[k,0]-scores[0,0]),architecture_change=float(scores[k,1]-scores[0,1]))
        rows.append(row)
results=pd.DataFrame(rows)
results.to_csv(OUT/'metrics.csv',index=False)
display(results.round(5))
summary=results.groupby('mode',sort=False)[['organic_gain','architecture_change','dino_patch_similarity','background_mae']].mean()
print('Means across 3 paired seeds (descriptive only):')
display(summary.round(5))
summary.to_csv(OUT/'summary.csv')
fig,axes=plt.subplots(3,4,figsize=(16,12))
for r,seed in enumerate(SEEDS):
    for c,mode in enumerate(MODES):
        ax=axes[r,c]; ax.imshow(baselines[seed] if mode=='baseline' else RUNS[(seed,mode)]['image']); ax.axis('off')
        gain=results.query('seed==@seed and mode==@mode').iloc[0].organic_gain
        ax.set_title(f'{mode} | seed {seed}\norganic delta {gain:+.4f}')
plt.tight_layout(); fig.savefig(OUT/'all_seeds_comparison.png',dpi=160,bbox_inches='tight'); plt.show()

In [ ]:
# 16. Numerical sanity check against the stock Diffusers pipeline
with torch.inference_mode():
    stock = pipe(BASE_PROMPT,negative_prompt=NEGATIVE,num_inference_steps=STEPS,guidance_scale=CFG,height=SIZE,width=SIZE,latents=initial_latent(17)).images[0]
stock_mae = float(np.abs(np.asarray(stock,dtype='float32')-np.asarray(baselines[17],dtype='float32')).mean()/255)
print('Custom baseline vs stock pipeline pixel MAE:',stock_mae)
assert stock_mae < .002, 'Custom sampler differs materially from stock baseline; inspect before interpreting.'
assert len(RUNS)==9 and len(baselines)==3
assert np.isfinite(results.select_dtypes('number').values).all()
assert len(API_LOG)==12, 'Expected 4 genuine TypeSafe decisions per seed.'
assert all(len(r['decisions'])==4 for r in RUNS.values())
print('Verified: stock sampler agreement, 12 images, 36 policy decisions, 12 real TypeSafe calls, finite metrics.')
# Archive only public model/config data and synthetic measurements.
config={'model':MODEL_ID,'dino':DINO_ID,'clip':CLIP_ID,'versions':VERSIONS,'seeds':SEEDS,'steps':STEPS,'size':SIZE,'cfg':CFG,'checkpoints':CHECKPOINTS,'horizon':HORIZON,'base_prompt':BASE_PROMPT,'bio_prompt':BIO_PROMPT,'negative':NEGATIVE,'actions':ACTION_NAMES,'strengths':ACTION_STRENGTHS.float().cpu().tolist(),'jev_questions':QUESTIONS,'routing_weights':{'material':.6,'structure':.25,'background':.15},'deterministic_utility':'100*organic_gain +30*min(architecture_change,0) -5*(1-dino_patch_similarity) -5*background_mae','scheduler':dict(pipe.scheduler.config)}
archive=json.dumps({'config':config,'typesafe_calls':API_LOG},indent=2,default=str)
assert _api_key not in archive
(OUT/'config_and_api_audit.json').write_text(archive)
print('API usage totals:',{k:sum(x.get('usage',{}).get(k,0) for x in API_LOG) for k in ['input_tokens','output_tokens']})
print('Saved:',OUT.resolve())

In [ ]:
# 17. Null-control replay: quantify floating-point drift from branch batching
null_rows=[]
for seed in SEEDS:
    z=baseline_paths[seed][9].clone(); i=9
    while i<STEPS:
        if i in CHECKPOINTS:
            zz=z.repeat(5,1,1,1)
            zero_masks=torch.zeros((5,1,64,64),device=DEVICE,dtype=DTYPE)
            for j in range(i,i+HORIZON): zz,_=advance(zz,j,zero_masks,ACTION_STRENGTHS)
            z=zz[:1].clone(); i+=HORIZON
        else: z,_=advance(z,i); i+=1
    img=decode(z)[0]; img.save(OUT/f'{seed}_null_replay.png')
    p,s=observe([baselines[seed],img]); _,m,_=find_regions(baselines[seed],p[0])
    nr=metrics(img,p[1],s[1],baselines[seed],p[0],m.sum(0,keepdim=True).clamp(0,1))
    nr.update(seed=seed,organic_gain=float(s[1,0]-s[0,0]),pixel_mae=float(np.abs(np.asarray(img,dtype='float32')-np.asarray(baselines[seed],dtype='float32')).mean()/255))
    null_rows.append(nr)
null_results=pd.DataFrame(null_rows)
null_results.to_csv(OUT/'null_replay_metrics.csv',index=False)
display(null_results.round(6))
print('This is the numerical floor; small intervention effects must be judged relative to it.')

In [ ]:
# 18. Interpret the result and preserve the complete audit trail
actions=pd.DataFrame([{'seed':seed,'policy':mode,'step':d['step'],'action':d['selected']} for (seed,mode),run in RUNS.items() for d in run['decisions']])
actions.to_csv(OUT/'actions.csv',index=False)
display(actions.pivot(index=['seed','policy'],columns='step',values='action'))
weak_choices=[]
for (seed,mode),run in RUNS.items():
    if mode!='jev': continue
    for d in run['decisions']:
        best=max(m['organic_gain'] for m in d['state']['candidate_outcomes'].values())
        if best<.001 and d['selected']!='none': weak_choices.append({'seed':seed,'step':d['step'],'selected':d['selected'],'best_organic_gain':best})
lines=['# Closed-loop diffusion pilot: observed results','',
'Ran SD1.5 at 512x512 with 36 DDIM steps on seeds 17, 42, 123. DINOv2-small with four registers supplied patch-region masks; CLIP supplied semantic proxy scores. At steps 9, 15, 21, 27, each controller measured five three-step futures from the same latent and committed one. Jev received goal, scene, regions, counterfactual measurements, and short history. It answered three narrow questions; code combined the probability distributions with weights 0.60/0.25/0.15.', '',
'## Mean final deltas relative to each seed baseline','',
'| Policy | Organic CLIP delta | Architecture CLIP delta | DINO similarity | Background MAE |',
'|---|---:|---:|---:|---:|']
for mode,row in summary.iterrows():
    lines.append(f'| {mode} | {row.organic_gain:+.5f} | {row.architecture_change:+.5f} | {row.dino_patch_similarity:.5f} | {row.background_mae:.5f} |')
lines += ['', '## Interpretation', '',
'The end-to-end loop works. Fixed guidance moves the biological proxy more, but changes structure and background more. Both controllers preserve the baseline better. Jev has slightly more biological gain and more drift than deterministic control; three seeds do not establish that Jev is better. Images remain predominantly cathedrals: this is not yet a convincing cathedral-organism transformation.',
'',f'The null branch replay has maximum absolute organic cosine drift {null_results.organic_gain.abs().max():.6f} and mean background MAE {null_results.background_mae.mean():.6f}. The stock Diffusers baseline comparison has pixel MAE {stock_mae:.6f}; float16 and batch shape produce small numerical differences.',
'',f'Jev selected an intervention despite all candidate gains being below the requested 0.001 evidence threshold at {len(weak_choices)} checkpoints. The threshold is a prompt preference, not a code-enforced constraint. This is a concrete reason to add numerical gating before trusting routing decisions.',
'', '## Limits and next experiment','',
'This is external spatial guidance, not internal diffusion-register steering. The intervention uses fixed conditional denoiser differences, not learned per-register causal directions. There is no trained causal atlas; live short-horizon probes supply local causal evidence. DINO clusters are heuristic regions and are recomputed without persistent object identity. We did not ablate DINO registers, remove context, compare compute-matched random policies, or collect independent human ratings. CLIP was used both to guide and evaluate, so its gain does not establish visual quality. Fixed guidance also uses a frozen region, while controllers use refreshed regions, so its comparison does not isolate the router.',
'', 'Next: enforce the weak-signal gate in code; use a stronger material direction and longer candidate lookahead; add dynamic-mask fixed/random controls and blinded ratings over more seeds. For the separate internal-register hypothesis, use a register-equipped diffusion transformer and ablate individual registers with phase-specific, same-seed controls.',
'', '## Files','', 'all_seeds_comparison.png, observer_regions.png, counterfactual_step9.png, metrics.csv, summary.csv, actions.csv, null_replay_metrics.csv, config_and_api_audit.json, individual PNGs and per-run decision JSON files. No credentials are included.']
report='\n'.join(lines)
assert _api_key not in report
(OUT/'README.md').write_text(report)
display(Markdown(report))

In [ ]:
# 19. Check spatial alignment of the observer preprocessing
print('DINO preprocessing:',{k:getattr(dino_processor,k,None) for k in ['size','crop_size','do_resize','do_center_crop']})
print('DINO input shape:',tuple(dino_processor(images=baselines[17],return_tensors='pt').pixel_values.shape))

In [ ]:
# 20. Correct spatial alignment; preserve the preliminary run separately
# The default DINO center crop excluded a 32px border of the 512px image.
# Resize the entire field of view to 224x224 before normalization instead.
PRELIMINARY_OUT=OUT
OUT=Path(str(PRELIMINARY_OUT)+'_aligned'); OUT.mkdir(exist_ok=True)
@torch.inference_mode()
def observe(images):
    full_view=[image.resize((224,224),Image.Resampling.BICUBIC) for image in images]
    pixels=dino_processor(images=full_view,do_resize=False,do_center_crop=False,return_tensors='pt').to(DEVICE)
    hidden=dino(**pixels).last_hidden_state
    patches=F.normalize(hidden[:,1+dino.config.num_register_tokens:],dim=-1)
    cf=F.normalize(clip.get_image_features(**clip_processor(images=images,return_tensors='pt').to(DEVICE)),dim=-1)
    scores=(cf@concept_features.T).cpu().numpy()
    return patches.cpu().numpy(),scores
for seed,img in baselines.items(): img.save(OUT/f'{seed}_baseline.png')
RUNS={}; API_LOG=[]
p9,s9=observe([preview9]); regions9,masks9,labels9=find_regions(preview9,p9[0])
branches9,previews9,state9,_=probe(z9,9,[])
first_jev_action,first_jev_audit=jev_route(state9)
print('Corrected full-view observer. Final results will be saved to:',OUT)
print('Earlier results remain archived as PRELIMINARY and are superseded.')
print('First corrected action:',first_jev_action)
display(pd.DataFrame(state9['candidate_outcomes']).T.round(5))

In [ ]:
# 21. Repeat the identical experimental design with correctly aligned masks
for seed in SEEDS:
    for mode in ['fixed','deterministic','jev']:
        run_control(seed,mode)
print('Corrected comparison complete: 3 matched seeds x 4 policies.')

In [ ]:
# 22. Final aligned measurements (supersede the preliminary tables)
rows=[]; null_rows=[]
for seed in SEEDS:
    images=[baselines[seed]]+[RUNS[(seed,m)]['image'] for m in MODES[1:]]
    patches,scores=observe(images); _,rm,_=find_regions(images[0],patches[0]); fg=rm.sum(0,keepdim=True).clamp(0,1)
    for k,mode in enumerate(MODES):
        row=metrics(images[k],patches[k],scores[k],images[0],patches[0],fg)
        row.update(seed=seed,mode=mode,organic_gain=float(scores[k,0]-scores[0,0]),architecture_change=float(scores[k,1]-scores[0,1])); rows.append(row)
    # Null latent trajectories do not depend on the observer: reuse their images, remeasure with corrected features.
    ni=Image.open(PRELIMINARY_OUT/f'{seed}_null_replay.png').convert('RGB'); ni.save(OUT/f'{seed}_null_replay.png')
    pn,sn=observe([ni]); nr=metrics(ni,pn[0],sn[0],images[0],patches[0],fg)
    nr.update(seed=seed,organic_gain=float(sn[0,0]-scores[0,0])); null_rows.append(nr)
results=pd.DataFrame(rows); results.to_csv(OUT/'metrics.csv',index=False)
null_results=pd.DataFrame(null_rows); null_results.to_csv(OUT/'null_replay_metrics.csv',index=False)
summary=results.groupby('mode',sort=False)[['organic_gain','architecture_change','dino_patch_similarity','background_mae']].mean()
summary.to_csv(OUT/'summary.csv')
display(results.round(5)); print('FINAL aligned means:'); display(summary.round(5))
fig,axes=plt.subplots(3,4,figsize=(16,12))
for r,seed in enumerate(SEEDS):
    for c,mode in enumerate(MODES):
        ax=axes[r,c]; ax.imshow(baselines[seed] if mode=='baseline' else RUNS[(seed,mode)]['image']); ax.axis('off')
        gain=results.query('seed==@seed and mode==@mode').iloc[0].organic_gain
        ax.set_title(f'{mode} | seed {seed}\norganic delta {gain:+.4f}')
plt.tight_layout(); fig.savefig(OUT/'all_seeds_comparison.png',dpi=160,bbox_inches='tight'); plt.show()
assert len(RUNS)==9 and len(API_LOG)==12 and np.isfinite(results.select_dtypes('number')).all().all()

In [ ]:
# 23. Save corrected observer figures, action logs, and exact API responses
fig,axes=plt.subplots(1,4,figsize=(15,4))
axes[0].imshow(preview9); axes[0].set_title('Step 9 predicted clean image')
axes[1].imshow(labels9,cmap='tab10'); axes[1].set_title('Aligned full-image DINO clusters')
for ax,m,name in zip(axes[2:],masks9,['Region A','Region B']):
    ax.imshow(preview9); ax.imshow(F.interpolate(m[None].float(),size=(SIZE,SIZE),mode='bilinear',align_corners=False)[0,0].cpu(),alpha=.5,cmap='magma',vmin=0,vmax=1); ax.set_title(name)
for ax in axes: ax.axis('off')
plt.tight_layout(); fig.savefig(OUT/'observer_regions.png',dpi=150,bbox_inches='tight'); plt.close(fig)
fig,axes=plt.subplots(1,5,figsize=(17,4))
for ax,name,img in zip(axes,ACTION_NAMES,previews9): ax.imshow(img); ax.set_title(name); ax.axis('off')
plt.tight_layout(); fig.savefig(OUT/'counterfactual_step9.png',dpi=150,bbox_inches='tight'); plt.close(fig)
actions=pd.DataFrame([{'seed':seed,'policy':mode,'step':d['step'],'action':d['selected']} for (seed,mode),run in RUNS.items() for d in run['decisions']])
actions.to_csv(OUT/'actions.csv',index=False)
config['dino_preprocessing']='Full image resized to 224x224 with PIL bicubic; do_resize=False, do_center_crop=False; ImageNet normalization from pretrained processor.'
archive=json.dumps({'config':config,'typesafe_calls':API_LOG},indent=2,default=str)
assert _api_key not in archive
(OUT/'config_and_api_audit.json').write_text(archive)
weak_choices=[{'seed':seed,'step':d['step'],'selected':d['selected']} for (seed,mode),run in RUNS.items() if mode=='jev' for d in run['decisions'] if d['selected']!='none' and max(m['organic_gain'] for m in d['state']['candidate_outcomes'].values())<.001]
print('Final actions:'); display(actions.pivot(index=['seed','policy'],columns='step',values='action'))
print('Weak-evidence TypeSafe interventions:',len(weak_choices))
print('Final-round API calls:',len(API_LOG),'| all-round calls:',24)
print('Aligned outputs:',OUT.resolve())

In [ ]:
# 24. Final report and downloadable result bundle
from IPython.display import HTML, Image as DisplayImage
import zipfile
table=['| Policy | Organic delta | Architecture delta | DINO similarity | Background MAE |','|---|---:|---:|---:|---:|']
for mode,row in summary.iterrows(): table.append(f'| {mode} | {row.organic_gain:+.5f} | {row.architecture_change:+.5f} | {row.dino_patch_similarity:.5f} | {row.background_mae:.5f} |')
final_report='\n'.join(['# FINAL: aligned closed-loop diffusion pilot','',
'These results supersede cells 15-18. The DINO center crop was detected in validation and replaced by a full-image resize in cell 20. All comparisons were rerun with aligned masks. Preliminary outputs are retained separately.','',
'SD1.5 at 512x512, 36 DDIM steps, seeds 17/42/123. DINOv2 with registers identifies spatial regions. At four checkpoints, five three-step counterfactuals are measured. Jev evaluates biological gain, structure and background separately; code combines its probability distributions and commits one measured branch.','',
'## Mean paired results across three seeds','']+table+['',
'## Interpretation','',
'The complete closed-loop experiment works. Jev moves the biological CLIP proxy about as much as fixed guidance, with less background drift. Deterministic control preserves the original most closely but barely increases biological character. The visible changes remain subtle and mostly architectural. These samples do not convincingly depict living cathedral organisms, and three seeds do not establish that Jev is superior.','',
f'Null replay maximum absolute organic drift: {null_results.organic_gain.abs().max():.6f}; mean background MAE: {null_results.background_mae.mean():.6f}. Custom-vs-stock sampler pixel MAE: {stock_mae:.6f}. Small numerical differences are present.','',
f'Jev chose an intervention despite all gains falling below the requested 0.001 threshold at {len(weak_choices)} final checkpoints. Enforce numerical thresholds in code in the next experiment.','',
'## Limits','',
'This tests register-assisted external spatial guidance, not internal diffusion-register steering. SD1.5 has no such registers. Fixed conditional denoiser differences supply actions; live short-horizon probes supply local causal evidence. There is no learned register atlas. DINO clusters are approximate regions without persistent object identities. CLIP is used for both control and evaluation. Fixed guidance freezes its region while controllers refresh theirs, so its comparison does not isolate routing. There are no register/context ablations, compute-matched random controls, independent human ratings, or statistical significance claims.','',
'## Next experiment','',
'Add a hard weak-signal gate, stronger material directions and longer lookahead. Compare dynamic-mask fixed/random policies and blinded ratings across more seeds. Test true register-specific causality separately using a register-equipped diffusion transformer.','',
'## Reproduction','',
'Run notebook cells in order with man.env beside the notebook; cell 20 starts the corrected round. Models remain loaded in GPU memory. Settings, versions, API questions/probabilities, individual images, region maps, counterfactual previews, decision JSONs, metrics.csv, summary.csv, actions.csv and null replay metrics are saved. No credentials are exported. Final round: 12 TypeSafe calls; preliminary round: 12 calls.'])
assert _api_key not in final_report
(OUT/'README.md').write_text(final_report)
(PRELIMINARY_OUT/'SUPERSEDED.txt').write_text('Use '+str(OUT)+' for final aligned results.')
zip_path=OUT.with_suffix('.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUT.iterdir()):
        if path.is_file(): bundle.write(path,arcname=OUT.name+'/'+path.name)
display(Markdown(final_report))
file_base='/files'+str(OUT.resolve())
display(HTML(f'<p><a href="{file_base}/all_seeds_comparison.png" target="_blank">Open full-resolution comparison</a> | <a href="/files{zip_path.resolve()}" download>Download all final results (ZIP)</a> | <a href="{file_base}/metrics.csv">Metrics CSV</a></p>'))
print('Final files:',len(list(OUT.iterdir())),'| bundle MiB:',round(zip_path.stat().st_size/2**20,2))

In [ ]:
# 25. Final visual result: three seeds, four policies
display(Markdown('### Final comparison — corrected full-image DINO masks\nColumns: **baseline → fixed guidance → deterministic controller → TypeSafe/Jev**. Rows: seeds **17, 42, 123**.'))
display(DisplayImage(filename=str(OUT/'all_seeds_comparison.png'),width=660))
display(HTML(f'<a href="{file_base}/all_seeds_comparison.png" target="_blank">Full-resolution image</a> | <a href="/files{zip_path.resolve()}" download>All results ZIP</a>'))